# Check 4: Schooling across seeds and social-rule ablations

This notebook addresses whether local social interactions consistently change alignment and group spacing, rather than drawing a conclusion from one favourable run.

The social model uses a repulsion-priority rule followed by the normalized **raw sum** of neighbour headings and unit vectors towards attraction-zone neighbours. No group centroid, polarization score, or desired collective-state label enters a fish's decision. The centroid is used only for measurement. This is a deliberately simplified implementation of the three-zone approach in [Couzin et al. (2002)](https://doi.org/10.1006/jtbi.2002.3065).

We use ten seeds and paired starting states for four conditions: all rules, no orientation, no attraction, and no social interactions. The reflecting arena remains active in every condition. This control matters because walls and chance alignment can affect the order parameters even without social interactions.

**Scope.** The current baseline has constant speed, 360° perception, instantaneous turns and no angular noise. It is not a complete replication of the golden-shiner model in [Puckett et al. (2018)](https://www.nature.com/articles/s41598-018-26037-9). Its supplement specifies additional perception, turning and noise parameters. These runs test implementation behaviour before adding light.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "shiner_model.py").is_file()
)
MODEL_DIR = PROJECT_ROOT
if str(MODEL_DIR) not in sys.path:
    sys.path.insert(0, str(MODEL_DIR))

from shiner_model import GoldenShinersModel, classify_collective_state
from checks.check_support import run_model, audit_run

## Experiment and why the comparison is informative

All conditions have 20 fish in the same 20 × 20 reflecting arena, speed 1, radii 0.5 / 3.0 / 5.5 and `dt=0.125`. Each run lasts 500 steps (62.5 simulated time units). The initial state is recorded at step 0.

We summarize the **last 100 records**, steps 401–500, to reduce the influence of initial random scattering. This is a descriptive late window, not proof of stationarity or an equilibrium distribution. For each seed, ablations start with exactly the same positions and headings. We assess polarization, rotation, nearest-neighbour distance and distance to the centroid together: high alignment alone does not establish a compact school.

In [2]:
SEEDS = list(range(10))
STEPS = 500
LATE_RECORDS = 100
CONDITIONS = {
    "All rules": ("repulsion", "orientation", "attraction"),
    "No orientation": ("repulsion", "attraction"),
    "No attraction": ("repulsion", "orientation"),
    "No social rules": (),
}
METRICS = ["Polarization", "Rotation", "MeanNND", "MeanDistanceToCentroid"]

def distance_component_sizes(points, radius):
    """Connected components of the undirected current social-range graph."""
    distances = np.linalg.norm(points[:, None] - points[None, :], axis=2)
    adjacency = distances <= radius
    np.fill_diagonal(adjacency, False)
    unseen = set(range(len(points)))
    sizes = []
    while unseen:
        frontier = [unseen.pop()]
        size = 0
        while frontier:
            current = frontier.pop()
            size += 1
            linked = set(np.flatnonzero(adjacency[current])) & unseen
            unseen.difference_update(linked)
            frontier.extend(linked)
        sizes.append(size)
    return sorted(sizes, reverse=True)


seed_results = []
connectivity_results = []
run_frames = {}
initial_states = {}
for condition, rules in CONDITIONS.items():
    for seed in SEEDS:
        case = run_model(n_fish=20, seed=seed, steps=STEPS, enabled_rules=rules)
        audit_run(case)
        group = case.datacollector.get_model_vars_dataframe()
        agents = case.datacollector.get_agent_vars_dataframe()
        initial = agents.xs(0, level="Step")
        if condition == "All rules":
            initial_states[seed] = initial.copy()
            final_points = agents.xs(STEPS, level="Step")[["X", "Y"]].to_numpy()
            component_sizes = distance_component_sizes(final_points, case.attraction_radius)
            assert sum(component_sizes) == len(final_points)
            connectivity_results.append({
                "Seed": seed,
                "Final components": len(component_sizes),
                "Largest component fraction": component_sizes[0] / len(final_points),
                "Component sizes": tuple(component_sizes),
            })
        else:
            pd.testing.assert_frame_equal(initial, initial_states[seed])
        run_frames[(condition, seed)] = group.copy()
        late = group.tail(LATE_RECORDS)
        result = {"Condition": condition, "Seed": seed}
        for metric in METRICS:
            result[metric] = late[metric].mean()
        result["Final polarization"] = group.iloc[-1]["Polarization"]
        result["Final group radius"] = group.iloc[-1]["MeanDistanceToCentroid"]
        result["Late polarized fraction"] = late["CollectiveState"].eq("polarized").mean()
        seed_results.append(result)

seed_results = pd.DataFrame(seed_results)
summary = seed_results.groupby("Condition", sort=False)[METRICS].agg(["mean", "std", "min", "max"])
print(f"Completed {len(seed_results)} paired runs; every recording/bounds/heading audit passed.")
print(f"Late window: steps {STEPS-LATE_RECORDS+1}–{STEPS}.")
summary.round(4)

Completed 40 paired runs; every recording/bounds/heading audit passed.
Late window: steps 401–500.


Polarization                         Rotation                  \
                        mean     std     min     max     mean     std     min   
Condition                                                                       
All rules             0.8723  0.1739  0.4564  0.9832   0.1461  0.1895  0.0070   
No orientation        0.1334  0.0264  0.1089  0.1972   0.0785  0.0327  0.0458   
No attraction         0.8506  0.1071  0.7123  1.0000   0.3028  0.2566  0.0399   
No social rules       0.2165  0.0776  0.0907  0.3166   0.1110  0.0583  0.0261   

                        MeanNND                          \
                    max    mean     std     min     max   
Condition                                                 
All rules        0.4704  0.6484  0.2183  0.5607  1.2687   
No orientation   0.1629  0.7295  0.0920  0.5545  0.8334   
No attraction    0.7742  1.0982  0.3482  0.5860  1.5438   
No social rules  0.2062  2.5260  0.1843  2.2640  2.8865   

                MeanDistanceToCentroid                          
                                  mean     std     min     max  
Condition                                                       
All rules                       2.2619  2.7304  1.0871  9.8646  
No orientation                  4.7772  1.6664  1.2497  6.8610  
No attraction                   3.2872  1.4814  1.3439  5.2198  
No social rules                 7.3814  0.5318  6.6600  8.3316

## Read the individual seeds as well as the average

The table below keeps seed-to-seed variation visible. The fraction of records labelled `polarized` is a secondary summary; continuous metrics remain the primary evidence. A run can align while remaining dispersed, or become compact without maintaining one direction.

In [3]:
seed_results[seed_results["Condition"].eq("All rules")].set_index("Seed").round(4)

,Condition,Polarization,Rotation,MeanNND,MeanDistanceToCentroid,Final polarization,Final group radius,Late polarized fraction
Seed,,,,,,,,
0,All rules,0.4564,0.4319,0.5840,9.8646,0.4477,5.5914,0.11
1,All rules,0.9423,0.0577,0.5744,1.1376,1.0000,1.1242,0.93
2,All rules,0.9433,0.4704,1.2687,1.9704,0.9837,2.2184,0.00
3,All rules,0.9681,0.0184,0.5850,1.1682,0.9996,1.2067,1.00
4,All rules,0.9389,0.0297,0.5971,1.2000,0.9998,1.1861,0.96
5,All rules,0.9832,0.0070,0.5638,1.0997,0.7436,1.0385,1.00
6,All rules,0.9380,0.0316,0.5607,1.0871,0.9999,1.0738,0.96
7,All rules,0.9327,0.0402,0.5746,1.1367,0.9996,1.1707,0.96
8,All rules,0.9646,0.0264,0.5931,1.1221,0.9999,1.1441,0.99


## Check whether the final school is interaction-connected

Small nearest-neighbour distance can occur in several separate clusters. To distinguish that from one connected school, construct an undirected graph from the **saved final positions**: an edge joins two fish when their distance is at most the outer social radius (5.5). This is the current omnidirectional interaction range, not an additional force or a fitted biological definition of a school.

More than one connected component means that the final configuration contains groups with no current social-range path between them. A single component establishes connectivity under this definition, not compactness, persistent cohesion or a stable dynamical state. This diagnostic observes the final record only.

In [4]:
connectivity = pd.DataFrame(connectivity_results).set_index("Seed")
connectivity

,Final components,Largest component fraction,Component sizes
Seed,,,
0,2,0.50,"(10, 10)"
1,1,1.00,"(20,)"
2,2,0.95,"(19, 1)"
3,1,1.00,"(20,)"
4,1,1.00,"(20,)"
5,1,1.00,"(20,)"
6,1,1.00,"(20,)"
7,1,1.00,"(20,)"
8,1,1.00,"(20,)"


## Paired differences and repeatability

Subtract each control from the full-rule run with the same seed. Positive polarization differences indicate additional alignment; negative group-radius differences indicate additional compactness. We report the number of seeds showing each direction, rather than requiring every stochastic initial state to behave identically. Repeating seed 0 also checks that the deterministic baseline reproduces its full saved trajectory.

In [5]:
paired_rows = []
full = seed_results[seed_results["Condition"].eq("All rules")].set_index("Seed")
for condition in CONDITIONS:
    if condition == "All rules":
        continue
    control = seed_results[seed_results["Condition"].eq(condition)].set_index("Seed")
    delta = full[METRICS] - control[METRICS]
    paired_rows.append({
        "Control": condition,
        "Mean polarization difference": delta["Polarization"].mean(),
        "Seeds with higher polarization": int(delta["Polarization"].gt(0).sum()),
        "Mean rotation difference": delta["Rotation"].mean(),
        "Mean NND difference": delta["MeanNND"].mean(),
        "Mean group-radius difference": delta["MeanDistanceToCentroid"].mean(),
        "Seeds with smaller group radius": int(delta["MeanDistanceToCentroid"].lt(0).sum()),
    })
paired_comparison = pd.DataFrame(paired_rows).set_index("Control")
repeated = run_model(n_fish=20, seed=0, steps=STEPS)
pd.testing.assert_frame_equal(
    repeated.datacollector.get_model_vars_dataframe(), run_frames[("All rules", 0)]
)
print("Identical initial states across conditions and exact seed-0 repeatability confirmed.")
paired_comparison.round(4)

Identical initial states across conditions and exact seed-0 repeatability confirmed.


,Mean polarization difference,Seeds with higher polarization,Mean rotation difference,Mean NND difference,Mean group-radius difference,Seeds with smaller group radius
Control,,,,,,
No orientation,0.7389,10,0.0676,-0.0812,-2.5153,9
No attraction,0.0217,5,-0.1567,-0.4498,-1.0253,9
No social rules,0.6558,10,0.0351,-1.8776,-5.1195,9


In [6]:
full_range = full["Polarization"].agg(["min", "max"])
no_social = paired_comparison.loc["No social rules"]
without_orientation = paired_comparison.loc["No orientation"]
without_attraction = paired_comparison.loc["No attraction"]
least_compact_seed = full["MeanDistanceToCentroid"].idxmax()
least_compact = full.loc[least_compact_seed]
final_connectivity = connectivity.loc[least_compact_seed]
if final_connectivity["Final components"] > 1:
    connectivity_note = (
        f"Its final interaction graph contains {int(final_connectivity['Final components'])} disconnected components "
        f"with sizes {final_connectivity['Component sizes']}; the final configuration is therefore not one interaction-connected group."
    )
else:
    connectivity_note = "Its final interaction graph is connected; that does not remove the late-window compactness limitation."
display(Markdown(f"""## What these outputs tell us

- The full model's late-window mean polarization ranges from **{full_range['min']:.3f} to {full_range['max']:.3f}** across the ten seeds. Variation across seeds is part of the result.
- Relative to no social interactions, the full model has higher late-window polarization in **{int(no_social['Seeds with higher polarization'])}/10** paired seeds and a smaller mean group radius in **{int(no_social['Seeds with smaller group radius'])}/10**. The mean radius difference is **{no_social['Mean group-radius difference']:.3f}** model distance units.
- Relative to removing orientation, the mean polarization difference is **{without_orientation['Mean polarization difference']:.3f}**. Relative to removing attraction, the mean radius difference is **{without_attraction['Mean group-radius difference']:.3f}**. These paired contrasts describe the contribution of each rule in this arena and observation window; they do not establish a universal causal effect for every density or parameter choice.
- Reliable whole-group compactness did **not** hold for every seed. Seed **{least_compact_seed}** has the largest late-window mean radius, **{least_compact['MeanDistanceToCentroid']:.3f}**, while its mean nearest-neighbour distance is only **{least_compact['MeanNND']:.3f}**. {connectivity_note} Local spacing alone would have hidden this distinction.
- Every run retained finite metrics, unit headings, valid positions and complete recordings. Identical initial states and exact repeatability rule out hidden reseeding or inconsistent start conditions in these comparisons.

**Before light:** examine the seed variation and ablations rather than selecting one attractive trajectory. The current tests do not establish milling stability, an optimal social-parameter set, or biological calibration. The next modelling decisions include perception, turning limits and social noise; the baseline currently implements none of those extensions. No environmental gradient or global group-control term was added here.
"""))

## What these outputs tell us

- The full model's late-window mean polarization ranges from **0.456 to 0.983** across the ten seeds. Variation across seeds is part of the result.
- Relative to no social interactions, the full model has higher late-window polarization in **10/10** paired seeds and a smaller mean group radius in **9/10**. The mean radius difference is **-5.119** model distance units.
- Relative to removing orientation, the mean polarization difference is **0.739**. Relative to removing attraction, the mean radius difference is **-1.025**. These paired contrasts describe the contribution of each rule in this arena and observation window; they do not establish a universal causal effect for every density or parameter choice.
- Reliable whole-group compactness did **not** hold for every seed. Seed **0** has the largest late-window mean radius, **9.865**, while its mean nearest-neighbour distance is only **0.584**. Its final interaction graph contains 2 disconnected components with sizes (10, 10); the final configuration is therefore not one interaction-connected group. Local spacing alone would have hidden this distinction.
- Every run retained finite metrics, unit headings, valid positions and complete recordings. Identical initial states and exact repeatability rule out hidden reseeding or inconsistent start conditions in these comparisons.

**Before light:** examine the seed variation and ablations rather than selecting one attractive trajectory. The current tests do not establish milling stability, an optimal social-parameter set, or biological calibration. The next modelling decisions include perception, turning limits and social noise; the baseline currently implements none of those extensions. No environmental gradient or global group-control term was added here.


## References and interpretation limits

[Couzin et al. (2002)](https://doi.org/10.1006/jtbi.2002.3065) motivates emergence from local interaction zones. [Tunstrøm et al. (2013)](https://doi.org/10.1371/journal.pcbi.1002915) motivates measuring both polarization and rotation. The model here uses raw per-step measurements; it does not reproduce their experimental temporal smoothing, sampling conditions or full state statistics.

[Katz et al. (2011)](https://doi.org/10.1073/pnas.1107583108) is a reason to treat explicit orientation matching as a modelling choice to test, rather than assuming that a high alignment score validates that rule for real shiners. These ablations probe the code's rule contributions; they are not a fitted inference of fish interactions.